# C-21 Monitoring and Drift Detection

**Control:** C-21 Monitoring & Drift Detection, MAK-AI Responsible AI Toolkit.

**Learning goal:** compare recent data with a reference period to see data drift (with the Kolmogorov-Smirnov test and the population stability index, written out in about ten lines) and performance drift (once true labels arrive); choose thresholds and an alert rule; and write a monitoring plan that says who looks, how often and what happens next.

**Time to run:** about 2 minutes on an ordinary laptop. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:** Bike Sharing Dataset (Fanaee-T, 2013), DOI 10.24432/C5W894, licence CC BY 4.0, hourly rental counts for 2011 and 2012, from https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset. The file holds 17,379 hourly rows (the dataset page lists 17,389). It is credited here because CC BY 4.0 asks for attribution.

**A caveat before you start:** this data has no people-groups. It describes a bike-share system in one city, whose demand grew by about two-thirds between 2011 and 2012, which makes drift easy to see. A model for people drifts more quietly. The notebook shows how to watch a fairness number per group in each window on your own data, and uses a stand-in column here only to show the pattern.

## Linked risk areas and questions

Generated from content 5.0.0. Do not edit by hand.

Risk areas: 3

- PO-01: Deployment Drift
- PO-02: Lack of Human Oversight
- PO-03: Accountability Gap

Questions: 19: Q-PP-20, Q-PP-21, Q-IP-24, Q-PO-01, Q-PO-02, Q-PO-03, Q-PO-04, Q-PO-05, Q-PO-07, Q-PO-16, Q-PO-17, Q-PO-18, Q-PO-11, Q-PO-12, Q-PO-14, Q-PO-20, Q-PO-21, Q-PO-22, Q-PO-23

## When to use this notebook

Use it when you are answering questions about how a model and its data hold up over time, before and after it is deployed, and your status is **Not yet** (especially with the reason code `ny_needs_skills`, "Needs skills or guidance the team does not have yet") or **Partly**. It helps with:

- `Q-PO-01` and `Q-PO-02`: have you set up continuous monitoring for performance degradation and data drift, and defined triggers and procedures for retraining?
- `Q-PO-17` and `Q-PO-18`: do you monitor fairness metrics in production, not just accuracy, and have you defined performance thresholds below which the system is flagged or switched off?
- `Q-IP-24`: have you tested the model's performance on data from different time periods?
- `Q-PP-20`: have you checked whether your training data still fits the period the model will be used in?

You need a table with a date on each row, the columns the model reads, and (for the performance part) the true outcome once it is known and the model's prediction.

In [ ]:
import hashlib
import importlib.metadata
import io
import platform
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fairlearn.metrics import MetricFrame
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

SEED = 20261030  # the one random seed; every random step below uses it
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 30)

PACKAGES = ['pandas', 'numpy', 'scikit-learn', 'scipy', 'fairlearn', 'matplotlib']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/275/data.csv'
DATA_SHA256 = 'e03de4ee4ef4dc376ac6e04bf829673c6269e8eba5c60fa121640fa2f829504f'
DATA_FILE = 'uci-bike-sharing-275-data.csv'  # the cache file name is fixed here

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    path = CACHE_DIR / file_name
    if path.exists():
        data, source, remedy = path.read_bytes(), f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data = response.read()
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    found = hashlib.sha256(data).hexdigest()
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + '.part')
        partial.write_bytes(data)
        partial.replace(path)  # the cache never holds a half-written file
    return data


# To use your own file, replace the next line with:  raw = Path('your-file.csv').read_bytes()
raw = fetch_checked(DATA_URL, DATA_SHA256, DATA_FILE)
DATA_SHA256_USED = hashlib.sha256(raw).hexdigest()  # the evidence summary records the hash of the bytes used
df = pd.read_csv(io.BytesIO(raw))
print(f'{len(df):,} rows, {df.shape[1]} columns; SHA-256 of the data used: {DATA_SHA256_USED[:12]}...')

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
DATE_COLUMN = 'dteday'                    # the column with the date of each row
REFERENCE_START = '2011-01-01'            # the reference period: the data the model was built on
REFERENCE_END = '2011-03-31'
WINDOW = 'M'                              # how recent data is cut up: M is calendar months, W is weeks
TARGET = 'cnt'                            # the true outcome, known some time after the row is recorded
NUMERIC_FEATURES = ['temp', 'hum', 'windspeed', 'hr']              # columns to watch for drift, measured as numbers
CATEGORY_FEATURES = ['weathersit', 'workingday', 'season']        # columns to watch for drift, made of categories
GROUP_COLUMN = 'workingday'               # a stand-in group for the fairness check; use your own groups
PREDICTION_COLUMN = None                  # name a column of your model's predictions to skip the practice model
PSI_WATCH = 0.10                          # population stability index: at or above this a feature goes on watch
PSI_ALERT = 0.25                          # at or above this a feature is in alert
KS_ALERT = 0.20                           # Kolmogorov-Smirnov statistic: at or above this a feature is in alert
ERROR_WATCH = 1.25                        # error in a window divided by the reference error: watch from here
ERROR_ALERT = 1.50                        # and alert from here
MIN_WINDOW_ROWS = 100                     # a window with fewer rows than this is not judged

### Step 1: the reference and the windows

**What this does.** The reference is the period the model was built on. Everything after it is cut into windows (here calendar months), and each window is compared with the reference. The practice model is trained on the reference only, as a real model would have been.

**What it shows.** How much data each window holds. Very small windows give unstable comparisons, which is why `MIN_WINDOW_ROWS` exists.

**What it does not show.** Whether the reference period was a good one to build on. If the reference is unusual (a holiday season, a pilot with a hand-picked group), every later window will look drifted for the wrong reason.

In [ ]:
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
in_reference = df[DATE_COLUMN].between(REFERENCE_START, REFERENCE_END)
reference = df[in_reference].copy()
recent = df[df[DATE_COLUMN] > REFERENCE_END].copy()
recent['window'] = recent[DATE_COLUMN].dt.to_period(WINDOW)
windows = {str(label): part for label, part in recent.groupby('window')}

print(f'reference: {REFERENCE_START} to {REFERENCE_END}, {len(reference):,} rows')
print(f'{len(windows)} windows, from {min(windows)} to {max(windows)}; rows per window {recent.groupby("window").size().min()} to {recent.groupby("window").size().max()}')

### Step 2: drift in the data, one feature at a time

**What this does.** For each feature and each window it computes two numbers against the reference:

- **Kolmogorov-Smirnov statistic** (numeric features): the largest gap between the two cumulative distributions. 0 means the same distribution and 1 means no overlap. With thousands of rows the test's p-value is almost always tiny even for small gaps, so look at the statistic, which says how big the difference is, not the p-value.
- **Population stability index (PSI)**: cut the feature into ten bands using the reference (for a category column, each category is counted on its own instead), and add up, over the bands, (share now minus share then) multiplied by the log of (share now divided by share then). Under 0.10 is usually read as stable, 0.10 to 0.25 as a moderate shift and above 0.25 as a large one. These cut-offs are rules of thumb from credit scoring, not laws. A category that never appeared in the reference makes the index very large, because a share of zero is replaced by a tiny number to avoid dividing by zero.

**What it shows.** Which features moved, when, and how much.

**What it does not show.** Whether the shift matters. A seasonal change in temperature is drift but may be harmless, and a tiny shift in one feature may be what breaks the model. Drift in the data is a warning to look at the model, not proof that it is wrong.

In [ ]:
def psi_from_shares(expected, actual):
    """Population stability index from two lists of shares that each add up to 1."""
    expected, actual = np.clip(np.asarray(expected, float), 1e-4, None), np.clip(np.asarray(actual, float), 1e-4, None)
    return float(np.sum((actual - expected) * np.log(actual / expected)))


def psi_numeric(reference_values, current_values, bands=10):
    edges = np.unique(np.quantile(reference_values, np.linspace(0, 1, bands + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    expected = np.histogram(reference_values, edges)[0] / len(reference_values)
    actual = np.histogram(current_values, edges)[0] / len(current_values)
    return psi_from_shares(expected, actual)


def psi_categories(reference_values, current_values):
    values = sorted(set(reference_values) | set(current_values))
    expected = [np.mean(reference_values == v) for v in values]
    actual = [np.mean(current_values == v) for v in values]
    return psi_from_shares(expected, actual)


psi_rows, ks_rows = {}, {}
for label, part in windows.items():
    if len(part) < MIN_WINDOW_ROWS:
        continue
    psi_rows[label] = {f: psi_numeric(reference[f].to_numpy(), part[f].to_numpy()) for f in NUMERIC_FEATURES}
    psi_rows[label].update({f: psi_categories(reference[f].to_numpy(), part[f].to_numpy()) for f in CATEGORY_FEATURES})
    ks_rows[label] = {f: ks_2samp(reference[f], part[f]).statistic for f in NUMERIC_FEATURES}

psi_table = pd.DataFrame(psi_rows).T
ks_table = pd.DataFrame(ks_rows).T
print('Population stability index by window (rows) and feature (columns):')
print(psi_table.round(3).to_string())
print()
print('Kolmogorov-Smirnov statistic by window and numeric feature:')
print(ks_table.round(3).to_string())

### Step 3: drift in performance, once the true outcome arrives

**What this does.** It trains a practice model on the reference period only (a gradient-boosted regression that predicts the outcome from the watched features), measures its typical error on part of the reference it did not train on, and then measures the same error in every later window. The **error ratio** is the window's average error divided by the reference error: 1 means the same as at the start, 2 means twice as wrong. The **mean miss** is the average of (prediction minus truth): below zero means the model is predicting too low. If you named `PREDICTION_COLUMN`, your own predictions are used instead.

**What it shows.** Whether the model still works, and whether it errs in one direction.

**What it does not show.** Why. It cannot tell a change in the world from a fault in the data pipeline. And you only see it when the true outcome arrives, which may be weeks after the prediction. Data drift can be seen immediately, performance drift only later, which is why you watch both. The reference error here is measured on hours picked at random from the reference period, in among the hours the model trained on, so it is lower than the error on a later stretch of time the model has never seen, and every error ratio reads somewhat high for that reason.

In [ ]:
FEATURES = NUMERIC_FEATURES + CATEGORY_FEATURES
if PREDICTION_COLUMN is None:
    train_part, hold_part = train_test_split(reference, test_size=0.3, random_state=SEED)
    model = HistGradientBoostingRegressor(random_state=SEED).fit(train_part[FEATURES], train_part[TARGET])
    predict = lambda part: model.predict(part[FEATURES])
    reference_error = mean_absolute_error(hold_part[TARGET], predict(hold_part))
else:
    predict = lambda part: part[PREDICTION_COLUMN].to_numpy()
    reference_error = mean_absolute_error(reference[TARGET], predict(reference))

performance_rows = {}
for label, part in windows.items():
    if len(part) < MIN_WINDOW_ROWS:
        continue
    predicted = predict(part)
    error = mean_absolute_error(part[TARGET], predicted)
    performance_rows[label] = {'rows': len(part), 'average error': error, 'error ratio': error / reference_error,
                               'mean miss': float(np.mean(predicted - part[TARGET]))}
performance = pd.DataFrame(performance_rows).T
print(f'reference error (average miss, in the units of {TARGET}): {reference_error:.1f}')
print(performance.round(2).to_string())

### Step 4: thresholds and an alert rule

**What this does.** It turns the numbers into one status per window with a rule written out in plain code:

- a window is **alert** if any watched feature has a PSI at or above `PSI_ALERT`, or a Kolmogorov-Smirnov statistic at or above `KS_ALERT`, or the error ratio is at or above `ERROR_ALERT`;
- it is **watch** if any feature has a PSI at or above `PSI_WATCH`, or the error ratio is at or above `ERROR_WATCH`;
- otherwise it is **ok**.

The numbers in the "Adapt here" cell are starting points, not recommendations. The "Stop and decide" cell after the charts asks you to choose; put the numbers you choose in the "Adapt here" cell and run the notebook again.

**What it shows.** How often the rule would have fired, and when. A rule that fires every month is ignored. One that never fires is useless.

**What it does not show.** The right thresholds. Those depend on the cost of a wrong prediction in your setting, on how many alerts someone can look at, and on how quickly the world changes. Try the rule on past data before trusting it.

In [ ]:
def status_for(label):
    worst_psi = psi_table.loc[label].max()
    worst_ks = ks_table.loc[label].max()
    ratio = performance.loc[label, 'error ratio']
    reasons = []
    if worst_psi >= PSI_ALERT: reasons.append(f'PSI {worst_psi:.2f} on {psi_table.loc[label].idxmax()}')
    if worst_ks >= KS_ALERT: reasons.append(f'KS {worst_ks:.2f} on {ks_table.loc[label].idxmax()}')
    if ratio >= ERROR_ALERT: reasons.append(f'error ratio {ratio:.2f}')
    if reasons:
        return 'alert', '; '.join(reasons)
    if worst_psi >= PSI_WATCH: reasons.append(f'PSI {worst_psi:.2f} on {psi_table.loc[label].idxmax()}')
    if ratio >= ERROR_WATCH: reasons.append(f'error ratio {ratio:.2f}')
    return ('watch', '; '.join(reasons)) if reasons else ('ok', '')


alerts = pd.DataFrame([dict(zip(['window', 'status', 'why'], (label, *status_for(label)))) for label in psi_table.index])
alerts = alerts.set_index('window')
print(alerts['status'].value_counts().to_string())
share_in_alert = (alerts['status'] == 'alert').mean()
if share_in_alert > 0.5:
    print(f'\nThe rule is in alert in {share_in_alert:.0%} of the windows. A rule that fires this often gets ignored: '
          'either the thresholds or the reference period do not fit this use. That is a result, not a fault.')
alerts

### Step 5: a fairness number per group, in each window

**What this does.** Fairlearn's `MetricFrame` computes any metric separately for each group. Run once per window, it shows whether the model's error drifts differently for different groups. Here the stand-in groups are working days and other days, because this data has no people. On your data, name the groups you care about in `GROUP_COLUMN` and use a metric that fits your use (for example recall, or the selection rate from C-13).

**What it shows.** A gap between groups that grows over time is a warning that the model is ageing unevenly, even when the overall error looks fine.

**What it does not show.** Whether a gap is large enough to act on. It also does not check how many rows each group has in a window: a group with few rows gives an unstable number, and this cell neither leaves it out nor warns about it. Small groups need the care shown in C-14.

**On your own data.** Keep the group columns separate from the model's inputs, record the group for each person at the time of the prediction, and decide before launch who looks at this table.

In [ ]:
group_rows = {}
for label, part in windows.items():
    if len(part) < MIN_WINDOW_ROWS:
        continue
    frame = MetricFrame(metrics=mean_absolute_error, y_true=part[TARGET], y_pred=predict(part),
                        sensitive_features=part[GROUP_COLUMN].astype(str))
    row = {f'error, group {g}': v for g, v in frame.by_group.items()}
    row['gap between groups'] = frame.difference()
    group_rows[label] = row
group_gaps = pd.DataFrame(group_rows).T
print(group_gaps.round(1).to_string())

### Step 6 (optional): the same data drift report with Evidently

**This cell is optional and is skipped in the automatic runs.** Evidently is a free tool that builds a drift report in one call. It is not part of this notebook's locked environment. To try it, install it in your own environment (`pip install evidently==0.7.23`), then run the cell. The cell was written from the tool's published description and has not been run by the authors of this notebook, so treat it as a starting point.

In [ ]:
from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset

definition = DataDefinition(numerical_columns=NUMERIC_FEATURES, categorical_columns=CATEGORY_FEATURES)
latest = windows[max(windows)]
report = Report([DataDriftPreset()])
snapshot = report.run(Dataset.from_pandas(latest[FEATURES], data_definition=definition),
                      Dataset.from_pandas(reference[FEATURES], data_definition=definition))
snapshot.save_html('evidently_drift_report.html')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
psi_table.max(axis=1).plot(ax=axes[0], marker='o', color='tab:blue')
axes[0].axhline(PSI_WATCH, linestyle='--', color='orange', label='watch')
axes[0].axhline(PSI_ALERT, linestyle='--', color='red', label='alert')
axes[0].set_title('Largest population stability index in each window')
axes[0].set_ylabel('PSI of the most drifted feature')
axes[0].legend()
performance['error ratio'].plot(ax=axes[1], marker='o', color='tab:blue')
axes[1].axhline(ERROR_WATCH, linestyle='--', color='orange', label='watch')
axes[1].axhline(ERROR_ALERT, linestyle='--', color='red', label='alert')
axes[1].set_title('Error in each window divided by the reference error')
axes[1].legend()
for axis in axes:
    axis.tick_params(axis='x', rotation=90)
fig.tight_layout()

### Stop and decide: the thresholds, and the monitoring plan

The notebook shows what the numbers do. It cannot decide how your team will act on them.

Questions to ask in the room:

1. How large a shift, in which features, would make you stop trusting the model? Who found the shift that matters most last time something went wrong?
2. How wrong can the model be before someone is harmed? What does a missed case, or a wrong flag, cost, and who pays?
3. How many alerts can someone really look at each month? A rule that fires every month will be ignored.
4. Who looks at this table, how often, and who do they tell? What happens when an alert fires: investigate, retrain, switch the model off, or tell the people affected?
5. Which groups will you watch, and what gap between them will you treat as an alert?

Write your choices below. Leave a field blank if you have not decided, and the plan will say so. The plan is written to a file you can attach.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
THRESHOLDS_WE_CHOSE = ''     # the numbers you will use and why, in your words
WHO_LOOKS = ''               # a role, not a name
HOW_OFTEN = ''               # for example: 'every month, in the first week'
WHAT_HAPPENS_ON_ALERT = ''   # who is told, what is checked first, who can switch the model off
GROUPS_WE_WATCH = ''         # the groups and the gap that counts as an alert

In [ ]:
EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# Small tables of numbers, one row per window. They hold no row of the data.
psi_table.round(4).to_csv(EVIDENCE_DIR / 'C-21_psi_by_window.csv')
ks_table.round(4).to_csv(EVIDENCE_DIR / 'C-21_ks_by_window.csv')
performance.round(4).to_csv(EVIDENCE_DIR / 'C-21_performance_by_window.csv')
alerts.to_csv(EVIDENCE_DIR / 'C-21_alerts_by_window.csv')
group_gaps.round(4).to_csv(EVIDENCE_DIR / 'C-21_group_gaps_by_window.csv')
fig.savefig(EVIDENCE_DIR / 'C-21_drift.png', dpi=100, bbox_inches='tight')

versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
plan = f"""## Monitoring plan
- Thresholds: {THRESHOLDS_WE_CHOSE or 'not written yet'}
- Who looks: {WHO_LOOKS or 'not written yet'}
- How often: {HOW_OFTEN or 'not written yet'}
- What happens on an alert: {WHAT_HAPPENS_ON_ALERT or 'not written yet'}
- Groups we watch and the gap that counts: {GROUPS_WE_WATCH or 'not written yet'}
"""
(EVIDENCE_DIR / 'C-21_monitoring_plan.md').write_text('# C-21 monitoring plan\n\n' + plan, encoding='utf-8')

summary = f"""# C-21 Monitoring and Drift Detection: evidence summary

Attach this file, with the CSV, PNG and plan files beside it, to the decision it supports (questions Q-PO-01, Q-PO-02, Q-PO-17, Q-PO-18, Q-IP-24, Q-PP-20). It holds numbers only, never the data.

## What was run
- Reference period: {REFERENCE_START} to {REFERENCE_END}, {len(reference):,} rows
- Windows: {len(performance)} (cut by '{WINDOW}'), windows under {MIN_WINDOW_ROWS} rows not judged
- Features watched: {', '.join(NUMERIC_FEATURES + CATEGORY_FEATURES)}
- Thresholds used: PSI watch {PSI_WATCH}, PSI alert {PSI_ALERT}, KS alert {KS_ALERT}, error ratio watch {ERROR_WATCH}, alert {ERROR_ALERT}
- Reference error: {reference_error:.2f}
- Model: {'practice gradient-boosted regression trained on the reference' if PREDICTION_COLUMN is None else 'predictions from the column ' + PREDICTION_COLUMN}

## Status of each window
{alerts.to_string()}

{plan}
## Files beside this summary
C-21_psi_by_window.csv, C-21_ks_by_window.csv, C-21_performance_by_window.csv, C-21_alerts_by_window.csv, C-21_group_gaps_by_window.csv, C-21_monitoring_plan.md, C-21_drift.png

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of the data file used: {DATA_SHA256_USED}
"""
(EVIDENCE_DIR / 'C-21_summary.md').write_text(summary, encoding='utf-8')
print(summary)

## What this does not show, and where to go next

- **Drift is not harm, and no drift is not safety.** A model can be badly wrong with stable inputs (the world changed in a way no input shows), and drift can be harmless. The numbers say where to look.
- **It watches what you give it.** Features or groups you do not record cannot be monitored. If you cannot tell which group a person belongs to after launch, you cannot watch a fairness number per group either. Decide that before launch.
- **Windows hide what happens inside them.** A short outage or a one-day event can vanish in a monthly average.
- **Rules of thumb are not standards.** The PSI cut-offs come from credit scoring. Set yours from your own costs.
- **True outcomes may arrive late or never.** If you only ever see the model's decisions, performance drift stays invisible. Plan how the truth will come back.

Where to go next: this control's resources on the platform (the SciPy page for `ks_2samp` and the Evidently documentation); C-13 Subgroup Fairness Metrics and C-14 Disaggregated Performance Analysis (which metric to watch per group); and the C-17 model card and C-28 Codebook & Deployment Documentation templates, for what to write down about the plan.